In [1]:
CELL_START_TIME = __import__('time').time()

import os
import sys
import warnings
import logging
import contextlib

# Quiet TensorFlow / CUDA logs as much as possible.
# Some low-level CUDA registration messages may still appear before TensorFlow logging is initialized.
# They are usually environment warnings, not model-code errors.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)


@contextlib.contextmanager
def suppress_stderr(enabled=True):
    if not enabled:
        yield
        return
    try:
        fd = sys.stderr.fileno()
    except Exception:
        yield
        return
    saved_fd = os.dup(fd)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), fd)
            yield
    finally:
        os.dup2(saved_fd, fd)
        os.close(saved_fd)

# =============================================================================
# LLM-TinyTransformer-IDS
# Proposed model: LLM-inspired tabular tokenization + Tiny Transformer for IDS.
# This is NOT TACNet reproduction and does not use any external LLM API.
# =============================================================================

# =============================================================================
# SECTION 0 - CONFIG
# =============================================================================
import glob
import time
import random

MODEL_NAME = "LLM-TinyTransformer-IDS"

# Use one selected part of CICIoT2023 for quick testing
DATASET_PATH = "/kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/train/train.csv"

DATASET_PATH_CANDIDATES = [
    DATASET_PATH,
    "/kaggle/input/CICIoT2023/CICIOT23/train/train.csv",
    "/kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/train/train.csv",
]

TARGET_COL = "label"

DROP_LEAKAGE_COLUMNS = [
    "label", "Label",
    "label_full", "label1", "label2", "label3", "label4",
    "timestamp", "timestamp_start", "timestamp_end",
    "device_name", "device_mac",
]

RESULTS_CSV = "llm_tiny_transformer_results.csv"
BEST_MODEL_PATH = "llm_tiny_transformer_best_model.keras"
REPORT_PATH = "llm_tiny_transformer_classification_report.txt"
TRAINING_CURVES_PATH = "llm_tiny_transformer_training_curves.png"
CONFUSION_MATRIX_PATH = "llm_tiny_transformer_confusion_matrix.png"

# Tiny configuration. This stays below the prohibited heavy settings.
TOKENIZER_MODE = "quantile"  # options: "uniform", "quantile"
POOLING_MODE = "gap_max"     # options: "gap", "gap_max", "cls"
NUM_BINS = 64
D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 192
NUM_TRANSFORMER_BLOCKS = 3
DROPOUT_RATE = 0.10
ATTN_DROPOUT = 0.05
LR = 5e-4
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 1024
MAX_EPOCHS = 50
PATIENCE = 5
USE_EARLY_STOPPING = True
RUN_TINY_SEARCH = False
PRINT_MODEL_SUMMARY = False
THRESHOLD_OBJECTIVE = "macro_f1"  # options: "weighted_f1", "macro_f1", "attack_f1"

TEST_SIZE = 0.20
VAL_SIZE = 0.20
RANDOM_SEED = 42

THRESHOLD_START = 0.10
THRESHOLD_END = 0.90
THRESHOLD_STEP = 0.005

if TOKENIZER_MODE not in {"uniform", "quantile"}:
    raise ValueError("TOKENIZER_MODE must be 'uniform' or 'quantile'.")
if POOLING_MODE not in {"gap", "gap_max", "cls"}:
    raise ValueError("POOLING_MODE must be 'gap', 'gap_max', or 'cls'.")
if THRESHOLD_OBJECTIVE not in {"weighted_f1", "macro_f1", "attack_f1"}:
    raise ValueError("THRESHOLD_OBJECTIVE must be 'weighted_f1', 'macro_f1', or 'attack_f1'.")
if not USE_EARLY_STOPPING:
    raise ValueError("USE_EARLY_STOPPING must remain True for the max-50-epoch experiment.")
if D_MODEL >= 128 or NUM_HEADS >= 8 or NUM_TRANSFORMER_BLOCKS >= 4:
    raise ValueError("Configuration is too large for the required Tiny Transformer setting.")

# =============================================================================
# SECTION 1 - IMPORTS AND REPRODUCIBILITY
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

with suppress_stderr(True):
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, Model, Input
    from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
tf.get_logger().setLevel("ERROR")

try:
    from absl import logging as absl_logging
    absl_logging.set_verbosity(absl_logging.ERROR)
except Exception:
    pass

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.backend.clear_session()

with suppress_stderr(True):
    gpus = tf.config.list_physical_devices("GPU")
print(f"  GPU detected                  : {len(gpus)}")

if not gpus:
    print("  [Warning] TensorFlow GPU not detected. Training may be slow.")
else:
    for gpu in gpus:
        with suppress_stderr(True):
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except Exception:
                pass


def progress(step, msg):
    print(f"[{step}/8] {msg}")


def fmt_metric(value):
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return "N/A"
    return f"{float(value):.4f}"


def round_metric(value, digits=4):
    if isinstance(value, (float, np.floating)) and np.isnan(value):
        return "N/A"
    return round(float(value), digits)


def format_class_distribution(counts):
    if len(counts) == 2:
        benign_count = 0
        attack_count = 0
        benign_found = False
        for label, count in counts.items():
            label_text = str(label).lower()
            if "benign" in label_text or "normal" in label_text:
                benign_found = True
                benign_count += int(count)
            else:
                attack_count += int(count)
        if not benign_found:
            benign_count = int(counts.iloc[0])
            attack_count = int(counts.iloc[1])
        return f"benign={benign_count:,} | attack={attack_count:,}"
    return " | ".join(f"{label}={int(count):,}" for label, count in counts.items())


def resolve_dataset_path():
    for candidate in DATASET_PATH_CANDIDATES:
        if candidate and os.path.exists(candidate):
            return candidate
    matches = sorted(glob.glob("/kaggle/input/**/combined_dataset.csv", recursive=True))
    if matches:
        return matches[0]
    raise FileNotFoundError(
        "Could not find combined_dataset.csv. Update DATASET_PATH to the Kaggle dataset file path."
    )

# =============================================================================
# SECTION 2 - LOAD DATASET
# =============================================================================
progress(1, "Loading dataset...")
PIPELINE_START = time.time()

resolved_dataset_path = resolve_dataset_path()
df = pd.read_csv(resolved_dataset_path)
if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Available columns: {list(df.columns)[:20]} ...")

df = df[df[TARGET_COL].notna()].reset_index(drop=True)
target_counts = df[TARGET_COL].value_counts()
task_type_preview = "Binary IDS" if len(target_counts) == 2 else "Multiclass IDS"

print(f"  Dataset path : {resolved_dataset_path}")
print(f"  Dataset shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print(f"  Target column: {TARGET_COL}")
print(f"  Task preview : {task_type_preview}")
print(f"  Distribution : {format_class_distribution(target_counts)}")

# =============================================================================
# SECTION 3 - PREPROCESSING WITH NO LEAKAGE
# Split order:
#   1. Raw train_val/test = 80/20, stratified.
#   2. Raw train/val = 80/20 of train_val, stratified.
#   3. Fit imputation, categorical maps, scaler only on X_train.
#   4. Transform val/test with train-fitted objects only.
# =============================================================================
progress(2, "Preprocessing with train-only fitted objects...")

drop_set = set(DROP_LEAKAGE_COLUMNS)
drop_set.add(TARGET_COL)
existing_drop = [c for c in df.columns if c in drop_set]
feature_cols = [c for c in df.columns if c not in drop_set]

print(f"  Dropped leakage/meta columns: {len(existing_drop)}")
print(f"  Remaining feature columns   : {len(feature_cols)}")

suspicious_terms = ["label", "target", "class", "timestamp"]
flagged_cols = [c for c in feature_cols if any(term in c.lower() for term in suspicious_terms)]
if flagged_cols:
    raise ValueError(
        "Leakage check failed. Suspicious columns still in features: "
        f"{flagged_cols}. Add them to DROP_LEAKAGE_COLUMNS."
    )
print("  Leakage safety check        : passed")

X_raw = df[feature_cols].copy()
y_raw = df[TARGET_COL].copy()

num_cols = X_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_raw.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"  Numeric/Categorical columns : {len(num_cols)}/{len(cat_cols)}")

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_
n_classes = len(class_names)
task_type = "Binary IDS" if n_classes == 2 else "Multiclass IDS"

X_train_val_raw, X_test_raw, y_train_val, y_test = train_test_split(
    X_raw,
    y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_encoded,
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_val_raw,
    y_train_val,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_train_val,
)

X_train_raw = X_train_raw.copy()
X_val_raw = X_val_raw.copy()
X_test_raw = X_test_raw.copy()
print(
    "  Split sizes                  : "
    f"train={len(X_train_raw):,} | val={len(X_val_raw):,} | test={len(X_test_raw):,}"
)

if num_cols:
    X_train_raw[num_cols] = X_train_raw[num_cols].replace([np.inf, -np.inf], np.nan)
    X_val_raw[num_cols] = X_val_raw[num_cols].replace([np.inf, -np.inf], np.nan)
    X_test_raw[num_cols] = X_test_raw[num_cols].replace([np.inf, -np.inf], np.nan)

if num_cols:
    train_medians = X_train_raw[num_cols].median(numeric_only=True).fillna(0)
    X_train_raw[num_cols] = X_train_raw[num_cols].fillna(train_medians)
    X_val_raw[num_cols] = X_val_raw[num_cols].fillna(train_medians)
    X_test_raw[num_cols] = X_test_raw[num_cols].fillna(train_medians)

train_modes = {}
for col in cat_cols:
    X_train_raw[col] = X_train_raw[col].astype("object")
    X_val_raw[col] = X_val_raw[col].astype("object")
    X_test_raw[col] = X_test_raw[col].astype("object")
    mode_values = X_train_raw[col].mode(dropna=True)
    train_modes[col] = mode_values.iloc[0] if not mode_values.empty else "unknown"

if cat_cols:
    X_train_raw[cat_cols] = X_train_raw[cat_cols].fillna(train_modes)
    X_val_raw[cat_cols] = X_val_raw[cat_cols].fillna(train_modes)
    X_test_raw[cat_cols] = X_test_raw[cat_cols].fillna(train_modes)

category_maps = {}
for col in cat_cols:
    train_values = X_train_raw[col].astype(str)
    classes = pd.Index(train_values.unique())
    mapping = {value: idx for idx, value in enumerate(classes)}
    category_maps[col] = mapping
    X_train_raw[col] = train_values.map(mapping).fillna(-1)
    X_val_raw[col] = X_val_raw[col].astype(str).map(mapping).fillna(-1)
    X_test_raw[col] = X_test_raw[col].astype(str).map(mapping).fillna(-1)

X_train_enc = X_train_raw.astype(np.float32)
X_val_enc = X_val_raw.astype(np.float32)
X_test_enc = X_test_raw.astype(np.float32)

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_enc)
X_val = scaler.transform(X_val_enc)
X_test = scaler.transform(X_test_enc)

X_train = np.clip(X_train, 0.0, 1.0).astype(np.float32)
X_val = np.clip(X_val, 0.0, 1.0).astype(np.float32)
X_test = np.clip(X_test, 0.0, 1.0).astype(np.float32)

n_features = X_train.shape[1]
class_counts = np.bincount(y_train, minlength=n_classes)
class_weight_dict = {
    i: (len(y_train) / (n_classes * class_counts[i])) if class_counts[i] > 0 else 0.0
    for i in range(n_classes)
}

print("  Scaling                       : MinMax fit on train only, then clipped to [0, 1]")
print(f"  Token sequence length         : {n_features} (= number of features)")
print(f"  Class weights                 : {class_weight_dict}")

# =============================================================================
# SECTION 4 - LLM-STYLE TOKENIZATION LAYER
# Each tabular feature becomes one token position.
# Discretization is fit on X_train only when TOKENIZER_MODE = "quantile".
# Final token = value_embedding(bin_id) + feature_embedding(feature_id)
#             + continuous_projection(raw_scaled_value)
# =============================================================================
progress(3, "Fitting LLM-style tabular tokenization...")


def fit_token_bin_edges(X_fit, num_bins, mode):
    if mode == "uniform":
        return [None] * X_fit.shape[1], np.ones(X_fit.shape[1], dtype=bool)

    quantile_points = np.linspace(0.0, 1.0, num_bins + 1, dtype=np.float64)[1:-1]
    bin_edges = []
    use_uniform = np.zeros(X_fit.shape[1], dtype=bool)

    for feature_idx in range(X_fit.shape[1]):
        values = X_fit[:, feature_idx]
        values = values[np.isfinite(values)]
        unique_values = np.unique(values)

        if unique_values.size < 4:
            bin_edges.append(None)
            use_uniform[feature_idx] = True
            continue

        edges = np.quantile(values, quantile_points).astype(np.float32)
        edges = np.unique(edges)
        edges = edges[np.isfinite(edges)]

        if edges.size < 2:
            bin_edges.append(None)
            use_uniform[feature_idx] = True
        else:
            bin_edges.append(edges)
            use_uniform[feature_idx] = False

    return bin_edges, use_uniform


def transform_to_token_ids(X, bin_edges, use_uniform, num_bins):
    token_ids = np.empty(X.shape, dtype=np.int32)
    uniform_ids = np.floor(X * num_bins).astype(np.int32)
    uniform_ids = np.clip(uniform_ids, 0, num_bins - 1)

    for feature_idx in range(X.shape[1]):
        edges = bin_edges[feature_idx]
        if use_uniform[feature_idx] or edges is None:
            token_ids[:, feature_idx] = uniform_ids[:, feature_idx]
        else:
            token_ids[:, feature_idx] = np.searchsorted(edges, X[:, feature_idx], side="right")

    token_ids = np.clip(token_ids, 0, num_bins - 1).astype(np.int32)
    return token_ids


bin_edges, uniform_fallback_mask = fit_token_bin_edges(X_train, NUM_BINS, TOKENIZER_MODE)
X_train_tokens = transform_to_token_ids(X_train, bin_edges, uniform_fallback_mask, NUM_BINS)
X_val_tokens = transform_to_token_ids(X_val, bin_edges, uniform_fallback_mask, NUM_BINS)
X_test_tokens = transform_to_token_ids(X_test, bin_edges, uniform_fallback_mask, NUM_BINS)

fallback_count = int(uniform_fallback_mask.sum())
print(f"  Tokenizer mode                : {TOKENIZER_MODE}")
print(f"  NUM_BINS                      : {NUM_BINS}")
print(f"  Uniform fallback features     : {fallback_count}/{n_features}")
print("  Bin edges fitted on           : X_train only")

@tf.keras.utils.register_keras_serializable(package="LLMTinyTransformerIDS")
class TabularTokenizerLayer(layers.Layer):
    """Embeds tabular bin IDs and scaled continuous values as token sequences."""

    def __init__(self, n_features, num_bins, d_model, dropout_rate=0.0, **kwargs):
        super().__init__(**kwargs)
        self.n_features = int(n_features)
        self.num_bins = int(num_bins)
        self.d_model = int(d_model)
        self.dropout_rate = float(dropout_rate)
        self.value_embedding = layers.Embedding(
            input_dim=self.num_bins,
            output_dim=self.d_model,
            name="value_embedding",
        )
        self.feature_embedding = layers.Embedding(
            input_dim=self.n_features,
            output_dim=self.d_model,
            name="feature_embedding",
        )
        self.continuous_projection = layers.Dense(
            self.d_model,
            activation=None,
            name="continuous_projection",
        )
        self.dropout = layers.Dropout(self.dropout_rate, name="token_dropout")

    def call(self, inputs, training=None):
        token_ids, scaled_values = inputs
        token_ids = tf.cast(token_ids, tf.int32)
        token_ids = tf.clip_by_value(token_ids, 0, self.num_bins - 1)
        scaled_values = tf.cast(scaled_values, tf.float32)

        value_emb = self.value_embedding(token_ids)
        feature_ids = tf.range(self.n_features, dtype=tf.int32)[tf.newaxis, :]
        feature_emb = self.feature_embedding(feature_ids)
        continuous_emb = self.continuous_projection(scaled_values[..., tf.newaxis])

        token_sequence = value_emb + feature_emb + continuous_emb
        return self.dropout(token_sequence, training=training)

    def get_config(self):
        config = super().get_config()
        config.update({
            "n_features": self.n_features,
            "num_bins": self.num_bins,
            "d_model": self.d_model,
            "dropout_rate": self.dropout_rate,
        })
        return config

# =============================================================================
# SECTION 5 - TINY TRANSFORMER MODEL
# Pre-norm block:
#   LayerNorm -> MultiHeadAttention -> Dropout -> residual
#   LayerNorm -> Dense(FF_DIM, gelu) -> Dropout -> Dense(D_MODEL)
#             -> Dropout -> residual
# =============================================================================
progress(4, "Building Tiny Transformer model...")

@tf.keras.utils.register_keras_serializable(package="LLMTinyTransformerIDS")
class CLSTokenLayer(layers.Layer):
    """Prepends a learnable CLS token for optional cls pooling."""

    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = int(d_model)

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name="cls_token",
            shape=(1, 1, self.d_model),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        cls_tokens = tf.tile(self.cls_token, [batch_size, 1, 1])
        return tf.concat([cls_tokens, x], axis=1)

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model})
        return config


@tf.keras.utils.register_keras_serializable(package="LLMTinyTransformerIDS")
class CLSPoolingLayer(layers.Layer):
    """Returns the first token from a sequence."""

    def call(self, x):
        return x[:, 0, :]


def transformer_block(x, d_model, num_heads, ff_dim, dropout_rate, attn_dropout, block_idx):
    prefix = f"block{block_idx}_"

    norm_1 = layers.LayerNormalization(epsilon=1e-6, name=prefix + "ln1")(x)
    attn_out = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=attn_dropout,
        name=prefix + "mha",
    )(norm_1, norm_1)
    attn_out = layers.Dropout(dropout_rate, name=prefix + "attn_dropout")(attn_out)
    x = layers.Add(name=prefix + "attn_residual")([x, attn_out])

    norm_2 = layers.LayerNormalization(epsilon=1e-6, name=prefix + "ln2")(x)
    ffn = layers.Dense(ff_dim, activation="gelu", name=prefix + "ffn_dense1")(norm_2)
    ffn = layers.Dropout(dropout_rate, name=prefix + "ffn_dropout1")(ffn)
    ffn = layers.Dense(d_model, activation=None, name=prefix + "ffn_dense2")(ffn)
    ffn = layers.Dropout(dropout_rate, name=prefix + "ffn_dropout2")(ffn)
    x = layers.Add(name=prefix + "ffn_residual")([x, ffn])

    return x


def build_llm_tiny_transformer_ids(n_features, n_classes):
    token_ids_input = Input(shape=(n_features,), dtype="int32", name="token_ids")
    scaled_values_input = Input(shape=(n_features,), dtype="float32", name="scaled_features")

    x = TabularTokenizerLayer(
        n_features=n_features,
        num_bins=NUM_BINS,
        d_model=D_MODEL,
        dropout_rate=DROPOUT_RATE,
        name="tabular_tokenizer",
    )([token_ids_input, scaled_values_input])

    if POOLING_MODE == "cls":
        x = CLSTokenLayer(D_MODEL, name="cls_token_layer")(x)

    for block_idx in range(NUM_TRANSFORMER_BLOCKS):
        x = transformer_block(
            x,
            d_model=D_MODEL,
            num_heads=NUM_HEADS,
            ff_dim=FF_DIM,
            dropout_rate=DROPOUT_RATE,
            attn_dropout=ATTN_DROPOUT,
            block_idx=block_idx,
        )

    x = layers.LayerNormalization(epsilon=1e-6, name="encoder_final_ln")(x)

    if POOLING_MODE == "gap":
        x = layers.GlobalAveragePooling1D(name="gap_pooling")(x)
    elif POOLING_MODE == "gap_max":
        avg_pool = layers.GlobalAveragePooling1D(name="gap_pooling")(x)
        max_pool = layers.GlobalMaxPooling1D(name="gmp_pooling")(x)
        x = layers.Concatenate(name="avg_max_pooling")([avg_pool, max_pool])
    else:
        x = CLSPoolingLayer(name="cls_pooling")(x)

    x = layers.Dense(D_MODEL, activation="gelu", name="head_dense")(x)
    x = layers.Dropout(DROPOUT_RATE, name="head_dropout")(x)
    output = layers.Dense(n_classes, activation="softmax", name="output")(x)

    return Model(
        inputs={"token_ids": token_ids_input, "scaled_features": scaled_values_input},
        outputs=output,
        name=MODEL_NAME,
    )


def make_optimizer():
    adamw_cls = getattr(keras.optimizers, "AdamW", None)
    if adamw_cls is not None:
        return adamw_cls(learning_rate=LR, weight_decay=WEIGHT_DECAY)
    return keras.optimizers.Adam(learning_rate=LR)


with suppress_stderr(True):
    model = build_llm_tiny_transformer_ids(n_features=n_features, n_classes=n_classes)
    model.compile(
        optimizer=make_optimizer(),
        loss="sparse_categorical_crossentropy",
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )

n_params = model.count_params()
print(f"  Model                         : {MODEL_NAME}")
print(f"  Task                          : {task_type}")
print(f"  n_features / n_classes        : {n_features} / {n_classes}")
print(f"  D_MODEL / HEADS / FF_DIM      : {D_MODEL} / {NUM_HEADS} / {FF_DIM}")
print(f"  Transformer blocks            : {NUM_TRANSFORMER_BLOCKS}")
print(f"  Pooling                       : {POOLING_MODE}")
print(f"  Dropout / Attention dropout   : {DROPOUT_RATE} / {ATTN_DROPOUT}")
print(f"  Trainable params              : {n_params:,}")
if PRINT_MODEL_SUMMARY:
    model.summary()

# =============================================================================
# SECTION 6 - TRAINING
# Test set is not used here. Best checkpoint and early stopping monitor val_loss.
# =============================================================================
progress(5, "Training...")

train_inputs = {"token_ids": X_train_tokens, "scaled_features": X_train}
val_inputs = {"token_ids": X_val_tokens, "scaled_features": X_val}
test_inputs = {"token_ids": X_test_tokens, "scaled_features": X_test}

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

MODEL_TRAIN_START = time.time()
with suppress_stderr(True):
    history = model.fit(
        train_inputs,
        y_train,
        validation_data=(val_inputs, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1,
    )
MODEL_TRAIN_END = time.time()
model_train_min = (MODEL_TRAIN_END - MODEL_TRAIN_START) / 60.0

epochs_stopped = len(history.history["loss"])
best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
best_val_loss = float(np.min(history.history["val_loss"]))
# Save the restored best weights after EarlyStopping.
model.save(BEST_MODEL_PATH)

print(f"  Epochs run                    : {epochs_stopped}/{MAX_EPOCHS}")
print(f"  Best epoch                    : {best_epoch}")
print(f"  Best val_loss                 : {best_val_loss:.4f}")
print(f"  Model training time           : {model_train_min:.2f} min")

# =============================================================================
# SECTION 7 - VALIDATION THRESHOLD TUNING AND TEST EVALUATION
# Binary threshold is tuned only on validation weighted F1, then applied once to test.
# =============================================================================
progress(6, "Tuning validation threshold and evaluating test set...")


def get_binary_roles(class_names):
    names = [str(name).lower() for name in class_names]
    benign_idx = None
    for idx, name in enumerate(names):
        if "benign" in name or "normal" in name:
            benign_idx = idx
            break
    if benign_idx is None:
        negative_idx = 0
        positive_idx = 1
    else:
        negative_idx = benign_idx
        positive_idx = 1 - benign_idx
    return positive_idx, negative_idx


def predict_with_threshold(y_prob, threshold, positive_idx, negative_idx):
    y_pred = np.full(y_prob.shape[0], negative_idx, dtype=np.int32)
    y_pred[y_prob[:, positive_idx] >= threshold] = positive_idx
    return y_pred


best_threshold = None
best_val_weighted_f1 = None
best_val_macro_f1 = None
best_val_attack_f1 = None
best_val_fpr = None
best_val_fnr = None
positive_class_idx = None
negative_class_idx = None

if n_classes == 2:
    positive_class_idx, negative_class_idx = get_binary_roles(class_names)
    with suppress_stderr(True):
        val_prob = model.predict(val_inputs, batch_size=BATCH_SIZE, verbose=0)
    thresholds = np.round(
        np.arange(THRESHOLD_START, THRESHOLD_END + (THRESHOLD_STEP / 2.0), THRESHOLD_STEP),
        3,
    )

    best_tuple = (-np.inf, -np.inf, -np.inf, -np.inf, 0.5)
    for threshold in thresholds:
        val_pred = predict_with_threshold(val_prob, threshold, positive_class_idx, negative_class_idx)
        val_weighted_f1 = f1_score(y_val, val_pred, average="weighted", zero_division=0)
        val_macro_f1 = f1_score(y_val, val_pred, average="macro", zero_division=0)
        val_attack_f1 = f1_score(
            (y_val == positive_class_idx).astype(np.int32),
            (val_pred == positive_class_idx).astype(np.int32),
            zero_division=0,
        )
        val_tp = int(((y_val == positive_class_idx) & (val_pred == positive_class_idx)).sum())
        val_tn = int(((y_val == negative_class_idx) & (val_pred == negative_class_idx)).sum())
        val_fp = int(((y_val == negative_class_idx) & (val_pred == positive_class_idx)).sum())
        val_fn = int(((y_val == positive_class_idx) & (val_pred == negative_class_idx)).sum())
        val_fpr = val_fp / max(val_fp + val_tn, 1)
        val_fnr = val_fn / max(val_fn + val_tp, 1)
        tie_distance = abs(float(threshold) - 0.5)

        if THRESHOLD_OBJECTIVE == "weighted_f1":
            candidate_tuple = (
                val_weighted_f1,
                val_attack_f1,
                val_macro_f1,
                -tie_distance,
                float(threshold),
            )
        elif THRESHOLD_OBJECTIVE == "macro_f1":
            candidate_tuple = (
                val_macro_f1,
                val_attack_f1,
                val_weighted_f1,
                -tie_distance,
                float(threshold),
            )
        else:
            candidate_tuple = (
                val_attack_f1,
                val_macro_f1,
                val_weighted_f1,
                -tie_distance,
                float(threshold),
            )

        if candidate_tuple > best_tuple:
            best_tuple = candidate_tuple
            best_threshold = float(threshold)
            best_val_weighted_f1 = float(val_weighted_f1)
            best_val_macro_f1 = float(val_macro_f1)
            best_val_attack_f1 = float(val_attack_f1)
            best_val_fpr = float(val_fpr)
            best_val_fnr = float(val_fnr)

    print(
        "  Selected threshold            : "
        f"{best_threshold:.3f} | objective={THRESHOLD_OBJECTIVE} "
        f"| val weighted F1={best_val_weighted_f1:.4f} "
        f"| val macro F1={best_val_macro_f1:.4f} "
        f"| val attack F1={best_val_attack_f1:.4f}"
    )
else:
    print("  Threshold tuning              : skipped for multiclass; using argmax")

# This is the first and only test-set prediction pass used for final reporting.
with suppress_stderr(True):
    y_prob = model.predict(test_inputs, batch_size=BATCH_SIZE, verbose=0)
if n_classes == 2:
    y_pred = predict_with_threshold(y_prob, best_threshold, positive_class_idx, negative_class_idx)
else:
    y_pred = np.argmax(y_prob, axis=1).astype(np.int32)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

try:
    if n_classes == 2:
        y_test_attack = (y_test == positive_class_idx).astype(np.int32)
        roc_auc = roc_auc_score(y_test_attack, y_prob[:, positive_class_idx])
    else:
        roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")
except Exception as exc:
    print(f"  ROC-AUC warning               : {exc}")
    roc_auc = float("nan")

try:
    if n_classes == 2:
        y_test_attack = (y_test == positive_class_idx).astype(np.int32)
        pr_auc = average_precision_score(y_test_attack, y_prob[:, positive_class_idx])
    else:
        y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))
        pr_auc = average_precision_score(y_test_bin, y_prob, average="weighted")
except Exception as exc:
    print(f"  PR-AUC warning                : {exc}")
    pr_auc = float("nan")

cm = confusion_matrix(y_test, y_pred, labels=list(range(n_classes)))
if n_classes == 2:
    tp = int(((y_test == positive_class_idx) & (y_pred == positive_class_idx)).sum())
    tn = int(((y_test == negative_class_idx) & (y_pred == negative_class_idx)).sum())
    fp = int(((y_test == negative_class_idx) & (y_pred == positive_class_idx)).sum())
    fn = int(((y_test == positive_class_idx) & (y_pred == negative_class_idx)).sum())
    fpr = fp / max(fp + tn, 1)
    fnr = fn / max(fn + tp, 1)
    attack_f1 = f1_score(
        (y_test == positive_class_idx).astype(np.int32),
        (y_pred == positive_class_idx).astype(np.int32),
        zero_division=0,
    )
else:
    tp = np.diag(cm)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp
    tn = cm.sum() - (tp + fp + fn)
    fpr = float(np.mean(fp / np.maximum(fp + tn, 1)))
    fnr = float(np.mean(fn / np.maximum(fn + tp, 1)))
    attack_f1 = float("nan")

print(f"  Test Acc                      : {fmt_metric(acc)}")
print(f"  Test F1                       : {fmt_metric(f1)}")
print(f"  Test ROC-AUC / PR-AUC         : {fmt_metric(roc_auc)} / {fmt_metric(pr_auc)}")
print(f"  Internal Macro F1             : {fmt_metric(macro_f1)}")
if n_classes == 2:
    print(f"  Internal Attack F1            : {fmt_metric(attack_f1)}")

# =============================================================================
# SECTION 8 - SAVE OUTPUTS IN BENCHMARK FORMAT
# CSV columns must remain exactly in this order.
# =============================================================================
progress(7, "Saving outputs...")

report_text = classification_report(
    y_test,
    y_pred,
    labels=list(range(n_classes)),
    target_names=[str(name) for name in class_names],
    zero_division=0,
)
with open(REPORT_PATH, "w") as report_file:
    report_file.write(report_text)
    if n_classes == 2:
        report_file.write("\n")
        report_file.write(f"Validation-selected threshold: {best_threshold:.3f}\n")
        report_file.write(f"Validation weighted F1: {best_val_weighted_f1:.4f}\n")
        report_file.write(f"Validation attack F1: {best_val_attack_f1:.4f}\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="Train Acc")
axes[0].plot(history.history["val_accuracy"], label="Val Acc")
axes[0].set_title("Accuracy over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_title("Loss over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True)

plt.suptitle(f"{MODEL_NAME} - Training Curves")
plt.tight_layout()
plt.savefig(TRAINING_CURVES_PATH, dpi=150)
plt.close()

plt.figure(figsize=(max(6, n_classes), max(5, min(12, n_classes))))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[str(name) for name in class_names],
    yticklabels=[str(name) for name in class_names],
)
plt.title(f"{MODEL_NAME} - Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=150)
plt.close()

CELL_END_TIME = time.time()
cell_runtime_min = (CELL_END_TIME - CELL_START_TIME) / 60.0
pipeline_min = (CELL_END_TIME - PIPELINE_START) / 60.0
train_time_str = f"{cell_runtime_min:.2f} min"

if n_classes == 2:
    threshold_note = f" validation threshold={best_threshold:.3f};"
else:
    threshold_note = " multiclass argmax evaluation;"

note = (
    "PROPOSED MODEL - NOT TACNet. "
    "LLM-inspired tabular tokenization + Tiny Transformer; "
    "no external LLM API; train-only preprocessing;"
    f" tokenizer={TOKENIZER_MODE}; pooling={POOLING_MODE};"
    f"{threshold_note} max_epoch={MAX_EPOCHS}; early stopping patience={PATIENCE}; "
    "Train Time=total cell runtime."
)

csv_columns = [
    "Paper",
    "Task Type",
    "Method Type",
    "Training Unit",
    "Acc",
    "Prec",
    "Recall",
    "F1",
    "ROC-AUC",
    "PR-AUC",
    "FPR",
    "FNR",
    "Train Time",
    "Params",
    "Comm Cost",
    "Training Steps",
    "Note",
]

results_df = pd.DataFrame([{
    "Paper": MODEL_NAME,
    "Task Type": task_type,
    "Method Type": "LLM-inspired Tokenization + Tiny Transformer",
    "Training Unit": "Epochs",
    "Acc": round_metric(acc),
    "Prec": round_metric(prec),
    "Recall": round_metric(rec),
    "F1": round_metric(f1),
    "ROC-AUC": round_metric(roc_auc),
    "PR-AUC": round_metric(pr_auc),
    "FPR": round_metric(fpr),
    "FNR": round_metric(fnr),
    "Train Time": train_time_str,
    "Params": int(n_params),
    "Comm Cost": "N/A",
    "Training Steps": int(epochs_stopped),
    "Note": note,
}], columns=csv_columns)

results_df.to_csv(RESULTS_CSV, index=False)

print(f"  Saved CSV                     : {RESULTS_CSV}")
print(f"  Saved best model              : {BEST_MODEL_PATH}")
print(f"  Saved classification report   : {REPORT_PATH}")
print(f"  Saved training curves         : {TRAINING_CURVES_PATH}")
print(f"  Saved confusion matrix        : {CONFUSION_MATRIX_PATH}")
print(f"  Cell runtime                  : {cell_runtime_min:.2f} min")
print(f"  Pipeline runtime              : {pipeline_min:.2f} min")

progress(8, "Done.")

print("\n" + "=" * 56)
print(f"Paper       : {MODEL_NAME}")
print(f"Acc         : {fmt_metric(acc)}")
print(f"Prec        : {fmt_metric(prec)}")
print(f"Recall      : {fmt_metric(rec)}")
print(f"F1          : {fmt_metric(f1)}")
print(f"ROC-AUC     : {fmt_metric(roc_auc)}")
print(f"PR-AUC      : {fmt_metric(pr_auc)}")
print(f"FPR         : {fmt_metric(fpr)}")
print(f"FNR         : {fmt_metric(fnr)}")
print(f"Train Time  : {train_time_str}")
print(f"Params      : {n_params:,}")
print("Comm Cost   : N/A")
print(f"Steps       : {epochs_stopped}")
print(f"Note        : {note}")
print("=" * 56)


E0000 00:00:1780273528.510108      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780273528.573710      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780273529.111649      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780273529.111692      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780273529.111695      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780273529.111697      23 computation_placer.cc:177] computation placer already registered. Please check linka

  GPU detected                  : 2
[1/8] Loading dataset...
  Dataset path : /kaggle/input/datasets/himadri07/ciciot2023/CICIOT23/train/train.csv
  Dataset shape: 5,491,971 rows x 47 columns
  Target column: label
  Task preview : Multiclass IDS
  Distribution : DDoS-ICMP_Flood=848,088 | DDoS-UDP_Flood=637,558 | DDoS-TCP_Flood=528,499 | DDoS-PSHACK_Flood=481,254 | DDoS-SYN_Flood=478,653 | DDoS-RSTFINFlood=475,441 | DDoS-SynonymousIP_Flood=422,083 | DoS-UDP_Flood=390,422 | DoS-TCP_Flood=314,174 | DoS-SYN_Flood=237,573 | BenignTraffic=129,538 | Mirai-greeth_flood=116,133 | Mirai-udpplain=104,814 | Mirai-greip_flood=88,821 | DDoS-ICMP_Fragmentation=53,046 | MITM-ArpSpoofing=36,316 | DDoS-UDP_Fragmentation=34,169 | DDoS-ACK_Fragmentation=33,581 | DNS_Spoofing=21,214 | Recon-HostDiscovery=15,737 | Recon-OSScan=11,587 | Recon-PortScan=9,648 | DoS-HTTP_Flood=8,487 | VulnerabilityScan=4,396 | DDoS-HTTP_Flood=3,371 | DDoS-SlowLoris=2,757 | DictionaryBruteForce=1,541 | BrowserHijacking=665 | Co

I0000 00:00:1780273614.244153      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1780273614.250056      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


  Model                         : LLM-TinyTransformer-IDS
  Task                          : Multiclass IDS
  n_features / n_classes        : 46 / 34
  D_MODEL / HEADS / FF_DIM      : 64 / 4 / 192
  Transformer blocks            : 3
  Pooling                       : gap_max
  Dropout / Attention dropout   : 0.1 / 0.05
  Trainable params              : 142,946
[5/8] Training...
Epoch 1/50


I0000 00:00:1780273628.919226      70 service.cc:152] XLA service 0x7cc858001c30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780273628.919265      70 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1780273628.919269      70 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1780273630.567876      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


   2/3433 ━━━━━━━━━━━━━━━━━━━━ 3:54 68ms/step - accuracy: 0.0063 - loss: 5.2934   

I0000 00:00:1780273642.270961      70 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3433/3433 ━━━━━━━━━━━━━━━━━━━━ 247s 65ms/step - accuracy: 0.8784 - loss: 1.2219 - val_accuracy: 0.9749 - val_loss: 0.0859 - learning_rate: 5.0000e-04
Epoch 2/50
3433/3433 ━━━━━━━━━━━━━━━━━━━━ 220s 64ms/step - accuracy: 0.9694 - loss: 0.8770 - val_accuracy: 0.9762 - val_loss: 0.0765 - learning_rate: 5.0000e-04
Epoch 3/50
3433/3433 ━━━━━━━━━━━━━━━━━━━━ 223s 65ms/step - accuracy: 0.9710 - loss: 0.8347 - val_accuracy: 0.9776 - val_loss: 0.0713 - learning_rate: 5.0000e-04
Epoch 4/50
3433/3433 ━━━━━━━━━━━━━━━━━━━━ 223s 65ms/step - accuracy: 0.9727 - loss: 0.8057 - val_accuracy: 0.9759 - val_loss: 0.0714 - learning_rate: 5.0000e-04
Epoch 5/50
3432/3433 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.9735 - loss: 0.7857
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
3433/3433 ━━━━━━━━━━━━━━━━━━━━ 223s 65ms/step - accuracy: 0.9733 - loss: 0.7887 - val_accuracy: 0.9756 - val_loss: 0.0751 - learning_rate: 5.0000e-04
Epoch 6/50
3433/3433 ━━━━━━━━━━━━━━━━━━━━ 223s 65ms/